In [21]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import polars as pl
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

RANDOM_SEED = 1000
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [27]:
from typing import Optional


def recall_at_k(y_true, y_score, k) -> float:
    top = np.argsort(y_score)[::-1][:k]
    n_true_positives_in_top_k = y_true[top].sum()
    n_true_positives_total = y_true.sum()
    return n_true_positives_in_top_k / n_true_positives_total


def load_and_clean_data(features_path, labels_path=None):
    df = pl.read_parquet(features_path)
    df = df.filter(pl.all_horizontal(pl.col("^.*_valid$")))

    if labels_path is not None:
        df_labels = pl.read_parquet(labels_path)
        df = df.join(df_labels, on="id")
        df = df.filter(pl.col("affinity_kcal_mol") < 0)

    return df


def evaluate_enrichment(
    y_true, y_prob, percents: list[float] = [0.01, 0.05, 0.10, 0.15]
):
    sorted_indices = np.argsort(y_prob)[::-1]
    total_actives = int(y_true.sum())

    results = []

    for pct in percents:
        k = max(1, int(len(y_true) * pct))
        top = sorted_indices[:k]

        prec = y_true[top].mean()
        rec = y_true[top].sum() / total_actives
        n_caught = int(y_true[top].sum())

        print(
            f"top {pct:.0%}: precision={prec:.3f} recall={rec:.3f} "
            f"n_active_caught={n_caught}/{total_actives}"
        )

        results.append({"Top %": pct * 100, "Precision": prec, "Recall": rec})

    return pl.DataFrame(results)


def plot_enrichment(df_results):
    df_melted = df_results.unpivot(
        index="Top %",
        on=["Precision", "Recall"],
        variable_name="Metric",
        value_name="Score",
    )

    plt.figure(figsize=(8, 5))
    sns.set_theme(style="whitegrid")

    ax = sns.lineplot(
        data=df_melted,
        x="Top %",
        y="Score",
        hue="Metric",
        marker="o",
        markersize=8,
        linewidth=2,
    )

    plt.title("Screening Enrichment: Precision vs. Recall", fontsize=14, pad=10)
    plt.xlabel("Top % of Ranked Molecules Screened", fontsize=12)
    plt.ylabel("Score (0.0 to 1.0)", fontsize=12)

    plt.ylim(0, 1.05)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}%"))

    plt.tight_layout()
    plt.show()


def prepare_evaluation_data(
    model: torch.nn.Module,
    x: dict[str, np.ndarray],
    y: np.ndarray,
    active_threshold: float,
    top_fraction: float = 0.01,
    device: Optional[torch.device] = None,
) -> tuple[np.ndarray, np.ndarray, int]:

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    y_binary = (y <= active_threshold).astype(int)

    dataset = FeatureDictDataset(x, y_binary.astype(np.float32))
    loader = DataLoader(dataset, batch_size=512, shuffle=False)

    model = model.to(device)
    model.eval()

    y_probs = []

    with torch.no_grad():
        for batch_x_dict, _ in loader:
            batch_x_dict = {k: v.to(device) for k, v in batch_x_dict.items()}
            logits = model(batch_x_dict)
            probs = torch.sigmoid(logits)
            y_probs.append(probs.cpu().numpy())

    p_active = np.vstack(y_probs).ravel()

    k = max(1, int(len(y_binary) * top_fraction))

    return y_binary, p_active, k

In [3]:
class FeatureDictDataset(Dataset):
    def __init__(self, x_dict: dict[str, np.ndarray], y: np.ndarray):
        self.x_dict = {k: torch.from_numpy(v) for k, v in x_dict.items()}
        self.y = torch.from_numpy(y).unsqueeze(1)
        self.keys = list(self.x_dict.keys())
        self.length = len(y)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        return {k: self.x_dict[k][idx] for k in self.keys}, self.y[idx]

In [4]:
class FeatureTower(nn.Module):
    def __init__(self, input_dim: int, embed_dim: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.GELU(),
            nn.Dropout(0.1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class MultiTowerSurrogate(nn.Module):
    def __init__(
        self,
        feature_dims: dict[str, int],
        embed_dims: dict[str, int],
        hidden_dims: list[int] = [256, 64],
    ):
        super().__init__()

        self.towers = nn.ModuleDict(
            {
                name: FeatureTower(dim, embed_dims.get(name, 32))
                for name, dim in feature_dims.items()
            }
        )

        cat_dim = sum(embed_dims.get(name, 32) for name in feature_dims.keys())

        layers = []
        prev_dim = cat_dim
        for h_dim in hidden_dims:
            layers.extend(
                [
                    nn.Linear(prev_dim, h_dim),
                    nn.BatchNorm1d(h_dim),
                    nn.GELU(),
                    nn.Dropout(0.2),
                ]
            )
            prev_dim = h_dim

        layers.append(nn.Linear(prev_dim, 1))
        self.head = nn.Sequential(*layers)

    def forward(self, x_dict: dict[str, torch.Tensor]) -> torch.Tensor:
        embeddings = [self.towers[name](x_dict[name]) for name in self.towers.keys()]
        x_cat = torch.cat(embeddings, dim=1)
        return self.head(x_cat)

In [5]:
def extract_features_and_target(
    df, feature_cols=None, target_col="affinity_kcal_mol"
) -> tuple[dict[str, np.ndarray], np.ndarray, list[str]]:
    if feature_cols is None:
        feature_cols = []

    x_dict = {}
    feature_names = []

    for col in feature_cols:
        arr = df[col].to_numpy()

        if arr.ndim == 1 and hasattr(arr[0], "__len__"):
            arr = np.stack(arr)

        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        x_dict[col] = arr.astype(np.float32)

        n_features = arr.shape[1]
        feature_names.extend([f"{col}_{i}" for i in range(n_features)])

    y = df[target_col].to_numpy()

    return x_dict, y, feature_names


def split_feature_dict(
    x_dict: dict[str, np.ndarray],
    y_binary: np.ndarray,
    train_size: float = 0.8,
    random_state: int = 1000,
) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray], np.ndarray, np.ndarray]:
    indices = np.arange(len(y_binary))

    train_idx, test_idx = train_test_split(
        indices, train_size=train_size, random_state=random_state, stratify=y_binary
    )

    x_train = {k: v[train_idx] for k, v in x_dict.items()}
    x_test = {k: v[test_idx] for k, v in x_dict.items()}

    y_train = y_binary[train_idx]
    y_test = y_binary[test_idx]

    return x_train, x_test, y_train, y_test

In [6]:
initial_features_file = "data/features/sample_initial_100k_features.parquet"
initial_labels_file = "data/docking_runs/sample_initial_100k.1L83.p2rank.1.parquet"

feature_cols = [
    "atom_pair",
    "autocorr",
    "descriptors",
    "ecfp",
    "functional_groups",
    "morse",
    "rdf",
    "topological_torsion",
    "usrcat",
    "whim",
]

df_initial_clean = load_and_clean_data(initial_features_file, initial_labels_file)
x_initial, y_initial, feature_names = extract_features_and_target(
    df_initial_clean, feature_cols, "affinity_kcal_mol"
)

assert y_initial is not None, "Target column missing; cannot calculate threshold."

active_threshold = np.percentile(y_initial, 1)
y_initial_binary = (y_initial <= active_threshold).astype(np.float32)

In [7]:
sum(y_initial_binary)

np.float32(943.0)

In [ ]:
x_initial_train, x_initial_test, y_initial_train, y_initial_test = split_feature_dict(
    x_dict=x_initial,
    y_binary=y_initial_binary,
    train_size=0.8,
    random_state=RANDOM_SEED,
)

In [9]:
feature_dims = {name: arr.shape[1] for name, arr in x_initial.items()}

custom_embed_sizes = {
    "ecfp": 128,
    "e3fp": 128,
    "atom_pair": 64,
    "topological_torsion": 64,
    "descriptors": 32,
    "autocorr": 32,
    "pharmacophore_3d": 32,
    "morse": 32,
    "rdf": 32,
    "electroshape": 16,
    "functional_groups": 16,
    "usrcat": 16,
}

In [ ]:
train_loader = DataLoader(
    FeatureDictDataset(x_initial_train, y_initial_train), batch_size=256, shuffle=True
)
test_loader = DataLoader(
    FeatureDictDataset(x_initial_test, y_initial_test), batch_size=512, shuffle=False
)

model = MultiTowerSurrogate(
    feature_dims=feature_dims, embed_dims=custom_embed_sizes, hidden_dims=[256, 64]
).to(device)

In [ ]:
num_pos = y_initial_train.sum()
pos_weight = torch.tensor(
    [(len(y_initial_train) - num_pos) / max(1, num_pos)], device=device
)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    for batch_x_dict, batch_y in train_loader:
        batch_x_dict = {k: v.to(device) for k, v in batch_x_dict.items()}
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x_dict)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

In [ ]:
model.eval()
y_probs = []

with torch.no_grad():
    for batch_x_dict, _ in test_loader:
        batch_x_dict = {k: v.to(device) for k, v in batch_x_dict.items()}
        probs = torch.sigmoid(model(batch_x_dict))
        y_probs.append(probs.cpu().numpy())

y_probs = np.vstack(y_probs).ravel()
df_metrics = evaluate_enrichment(y_initial_test, y_probs)

In [12]:
holdout_features_file = "data/features/sample_holdout_100k_features.parquet"
holdout_labels_file = "data/docking_runs/sample_holdout_100k.1L83.p2rank.1.parquet"

df_holdout_clean = load_and_clean_data(holdout_features_file, holdout_labels_file)
x_holdout, y_holdout, feature_names = extract_features_and_target(
    df_holdout_clean, feature_cols, "affinity_kcal_mol"
)

y_holdout_binary = (y_holdout <= active_threshold).astype(np.float32)

In [ ]:
holdout_dataset = FeatureDictDataset(x_holdout, y_holdout_binary)
holdout_loader = DataLoader(holdout_dataset, batch_size=512, shuffle=False)

model.eval()
y_holdout_probs = []

with torch.no_grad():
    for batch_x_dict, _ in holdout_loader:
        batch_x_dict = {k: v.to(device) for k, v in batch_x_dict.items()}
        logits = model(batch_x_dict)
        probs = torch.sigmoid(logits)
        y_holdout_probs.append(probs.cpu().numpy())

y_holdout_probs = np.vstack(y_holdout_probs).ravel()

NameError: name 'model' is not defined

In [ ]:
df_metrics = evaluate_enrichment(y_holdout_binary, y_holdout_probs)

In [13]:
def update_training_arrays(
    x_train_current: dict[str, np.ndarray],
    y_train_current: np.ndarray,
    x_acquired: dict[str, np.ndarray],
    y_acquired_raw: np.ndarray,
    active_threshold: float,
) -> tuple[dict[str, np.ndarray], np.ndarray]:
    y_acquired_binary = (y_acquired_raw <= active_threshold).astype(np.float32)
    y_train_combined = np.concatenate([y_train_current, y_acquired_binary])

    x_train_combined = {}
    for key in x_train_current.keys():
        x_train_combined[key] = np.concatenate(
            [x_train_current[key], x_acquired[key]], axis=0
        )

    return x_train_combined, y_train_combined

In [14]:
active_round_1_features_file = "data/features/active_round_1.parquet"
active_round_1_labels_file = "data/docking_runs/active_round_1.1L83.p2rank.1.parquet"

df_active_round_1_clean = load_and_clean_data(
    active_round_1_features_file, active_round_1_labels_file
)
x_active_round_1, y_active_round_1, feature_names = extract_features_and_target(
    df_active_round_1_clean, feature_cols, "affinity_kcal_mol"
)

In [15]:
x_active_round_1, y_active_round_1 = update_training_arrays(
    x_initial, y_initial_binary, x_active_round_1, y_active_round_1, active_threshold
)

In [17]:
combined_train_dataset = FeatureDictDataset(x_active_round_1, y_active_round_1)
combined_train_loader = DataLoader(combined_train_dataset, batch_size=256, shuffle=True)

In [22]:
active_model_256_64 = MultiTowerSurrogate(
    feature_dims=feature_dims, embed_dims=custom_embed_sizes, hidden_dims=[256, 64]
).to(device)

In [23]:
import time
from typing import Optional


def train_surrogate_model(
    model: nn.Module,
    dataset: FeatureDictDataset,
    epochs: int = 10,
    batch_size: int = 2,
    lr: float = 1e-3,
    device: Optional[torch.device] = None,
) -> nn.Module:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    num_pos = dataset.y.sum().item()
    num_neg = len(dataset) - num_pos
    weight_ratio = num_neg / max(1.0, float(num_pos))
    pos_weight = torch.tensor([weight_ratio], dtype=torch.float32, device=device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    print(f"--- Starting Training on {device} ---")
    print(
        f"Total Samples: {len(dataset)} | Actives: {int(num_pos)}"
        f" | Pos-Weight: {weight_ratio:.2f}"
    )

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        start_time = time.time()

        for batch_x_dict, batch_y in loader:
            batch_x_dict = {k: v.to(device) for k, v in batch_x_dict.items()}
            batch_y = batch_y.to(device).float()

            optimizer.zero_grad()
            logits = model(batch_x_dict)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch_y.size(0)

        avg_loss = total_loss / len(dataset)
        epoch_time = time.time() - start_time

        print(
            f"Epoch [{epoch:02d}/{epochs:02d}]"
            f" - Loss: {avg_loss:.4f} - Time: {epoch_time:.2f}s"
        )

    return model

In [25]:
active_model_256_64_trained = train_surrogate_model(
    model=active_model_256_64,
    dataset=combined_train_dataset,
    epochs=20,
    batch_size=256,
    lr=1e-3,
    device=device,
)

--- Starting Training on cuda ---
Total Samples: 185983 | Actives: 40132 | Pos-Weight: 3.63
Epoch [01/20] - Loss: 0.1854 - Time: 7.15s
Epoch [02/20] - Loss: 0.1651 - Time: 7.27s
Epoch [03/20] - Loss: 0.1518 - Time: 7.39s
Epoch [04/20] - Loss: 0.1394 - Time: 7.02s
Epoch [05/20] - Loss: 0.1307 - Time: 7.26s
Epoch [06/20] - Loss: 0.1260 - Time: 7.04s
Epoch [07/20] - Loss: 0.1176 - Time: 7.84s
Epoch [08/20] - Loss: 0.1087 - Time: 7.67s
Epoch [09/20] - Loss: 0.1050 - Time: 7.41s
Epoch [10/20] - Loss: 0.1007 - Time: 7.40s
Epoch [11/20] - Loss: 0.0948 - Time: 7.64s
Epoch [12/20] - Loss: 0.0929 - Time: 7.26s
Epoch [13/20] - Loss: 0.0897 - Time: 7.21s
Epoch [14/20] - Loss: 0.0849 - Time: 7.47s
Epoch [15/20] - Loss: 0.0815 - Time: 7.01s
Epoch [16/20] - Loss: 0.0802 - Time: 7.13s
Epoch [17/20] - Loss: 0.0805 - Time: 7.02s
Epoch [18/20] - Loss: 0.0768 - Time: 7.31s
Epoch [19/20] - Loss: 0.0750 - Time: 7.61s
Epoch [20/20] - Loss: 0.0731 - Time: 6.99s


In [28]:
y_holdout_active_round_1_binary, p_holdout_active_round_1, k_holdout_active_round_1 = (
    prepare_evaluation_data(
        model=active_model_256_64_trained,
        x=x_holdout,
        y=y_holdout,
        active_threshold=active_threshold,
    )
)

In [29]:
df_holdout_active_round_1_metrics = evaluate_enrichment(
    y_holdout_active_round_1_binary,
    p_holdout_active_round_1,
    [0.01, 0.02, 0.03, 0.04, 0.05, 0.1],
)

top 1%: precision=0.428 recall=0.441 n_active_caught=401/909
top 2%: precision=0.267 recall=0.550 n_active_caught=500/909
top 3%: precision=0.207 recall=0.639 n_active_caught=581/909
top 4%: precision=0.171 recall=0.704 n_active_caught=640/909
top 5%: precision=0.146 recall=0.750 n_active_caught=682/909
top 10%: precision=0.082 recall=0.848 n_active_caught=771/909


In [30]:
active_model_512_128 = MultiTowerSurrogate(
    feature_dims=feature_dims, embed_dims=custom_embed_sizes, hidden_dims=[512, 128]
).to(device)

In [31]:
active_model_512_128_trained = train_surrogate_model(
    model=active_model_512_128,
    dataset=combined_train_dataset,
    epochs=20,
    batch_size=256,
    lr=1e-3,
    device=device,
)

--- Starting Training on cuda ---
Total Samples: 185983 | Actives: 40132 | Pos-Weight: 3.63
Epoch [01/20] - Loss: 0.5767 - Time: 7.61s
Epoch [02/20] - Loss: 0.5292 - Time: 7.92s
Epoch [03/20] - Loss: 0.5015 - Time: 7.42s
Epoch [04/20] - Loss: 0.4601 - Time: 7.38s
Epoch [05/20] - Loss: 0.4053 - Time: 7.39s
Epoch [06/20] - Loss: 0.3474 - Time: 7.26s
Epoch [07/20] - Loss: 0.2969 - Time: 7.73s
Epoch [08/20] - Loss: 0.2562 - Time: 7.21s
Epoch [09/20] - Loss: 0.2268 - Time: 7.20s
Epoch [10/20] - Loss: 0.1991 - Time: 7.75s
Epoch [11/20] - Loss: 0.1795 - Time: 7.21s
Epoch [12/20] - Loss: 0.1637 - Time: 7.42s
Epoch [13/20] - Loss: 0.1504 - Time: 6.92s
Epoch [14/20] - Loss: 0.1394 - Time: 7.03s
Epoch [15/20] - Loss: 0.1295 - Time: 7.58s
Epoch [16/20] - Loss: 0.1218 - Time: 7.22s
Epoch [17/20] - Loss: 0.1133 - Time: 6.99s
Epoch [18/20] - Loss: 0.1110 - Time: 6.82s
Epoch [19/20] - Loss: 0.1015 - Time: 7.25s
Epoch [20/20] - Loss: 0.0971 - Time: 6.84s


In [32]:
(
    y_holdout_active_round_1_binary_512_128,
    p_holdout_active_round_1_512_128,
    k_holdout_active_round_1_512_128,
) = prepare_evaluation_data(
    model=active_model_512_128_trained,
    x=x_holdout,
    y=y_holdout,
    active_threshold=active_threshold,
)

In [33]:
df_holdout_active_round_1_metrics_512_128 = evaluate_enrichment(
    y_holdout_active_round_1_binary_512_128,
    p_holdout_active_round_1_512_128,
    [0.01, 0.02, 0.03, 0.04, 0.05, 0.1],
)

top 1%: precision=0.417 recall=0.430 n_active_caught=391/909
top 2%: precision=0.267 recall=0.551 n_active_caught=501/909
top 3%: precision=0.208 recall=0.642 n_active_caught=584/909
top 4%: precision=0.173 recall=0.712 n_active_caught=647/909
top 5%: precision=0.146 recall=0.754 n_active_caught=685/909
top 10%: precision=0.082 recall=0.848 n_active_caught=771/909
